# HW1 — Financial Data Audit & Leakage-Free Feature Engineering

**FIN 7057 · Due after Week 3 · 100 points**

Fill in every `# TODO`. The notebook must run top-to-bottom with **Restart & Run All**.
See `README.md` for the full rubric. Part C (leakage-free features) is weighted most heavily.


## 0. Setup & reproducibility (required)


In [ ]:
# !pip install -q yfinance statsmodels scipy scikit-learn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

SEED = 42
np.random.seed(SEED)

# TODO: choose your instrument and EXACT date range, and document the source.
TICKER = 'SPY'
START, END = '2010-01-01', '2024-12-31'
DATA_SOURCE = 'Yahoo Finance via yfinance (auto-adjusted close)'   # TODO: edit if different


In [ ]:
# Provided loader (real data, with a synthetic fallback so the notebook always runs).
def load_prices(ticker, start, end):
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if len(df):
            s = df['Close'].squeeze(); s.name='close'; s.index.name='date'
            print(f'Loaded {len(s)} real rows for {ticker}.'); return s.to_frame()
        raise RuntimeError('empty')
    except Exception as e:
        print(f'Using synthetic fallback ({e!r}). Document this if you submit it.')
        n=252*15; dates=pd.bdate_range(start=start, periods=n)
        rng=np.random.default_rng(SEED); r=np.zeros(n); sig=np.zeros(n); sig[0]=0.01
        for t in range(1,n):
            sig[t]=np.sqrt(2e-6+0.10*r[t-1]**2+0.88*sig[t-1]**2); r[t]=sig[t]*rng.standard_t(5)/np.sqrt(5/3)
        return pd.DataFrame({'close':100*np.exp(np.cumsum(r))}, index=pd.Index(dates,name='date'))

px = load_prices(TICKER, START, END)
px.head()


## Part A — Data acquisition & audit (25 pts)


In [ ]:
# A2: returns. TODO: compute simple and log returns; keep the one you'll use.
px['simple_ret'] = None  # TODO
px['log_ret']    = None  # TODO

# A3: audit. TODO: report missing values, duplicate dates, calendar gaps, non-positive prices, outliers.
#   e.g., px.isna().sum(), px.index.duplicated().sum(), (px['close']<=0).sum() ...

# A4: summary statistics table. TODO.


*A1/A3 written answers:* TODO — document the data source, date range, and what your audit found.


## Part B — Time-series diagnostics (20 pts)
Run each test and **interpret it in one sentence**.


In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

# B1: ADF on price vs returns. TODO: run adfuller on each and interpret.
# B2: ACF of returns vs squared returns. TODO: plot_acf for each.
# B3: heavy tails. TODO: stats.kurtosis, stats.jarque_bera, stats.probplot (Q-Q).
# B4: rolling annualized volatility plot. TODO.


*B interpretations:* TODO — one sentence per test (stationarity, vol clustering, heavy tails).


## Part C — Leakage-free feature engineering (35 pts)

Write `build_features(prices)` returning a DataFrame of **feature columns only** (no target).
Every feature must use only information available **at or before the prior day**. Then run the
provided `assert_no_lookahead` check — it recomputes your features on a truncated history and
verifies the past values don't change. Past-only features ⇒ discrepancy ≈ 0.

Before fitting anything, add two compact Markdown tables:
1. A **feature contract** for every column: source, availability, transformation, memory, unit, hypothesis.
2. A **target contract**: decision time, execution assumption, horizon, return convention, missing-label policy.


In [ ]:
def build_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame of features (X only). Use ONLY past data (shift!)."""
    p = prices.copy()
    p['log_ret'] = np.log(p['close']).diff()
    f = pd.DataFrame(index=p.index)
    # Example of a correct, past-only feature:
    f['ret_lag1'] = p['log_ret'].shift(1)
    # TODO: add >=6 features total across families:
    #   - rolling momentum over >=2 windows (e.g., p['log_ret'].rolling(W).mean().shift(1))
    #   - rolling volatility
    #   - a normalized / z-scored feature
    #   - one of your choice (RSI, high-low range, calendar dummy, ...)
    # Remember: to compute feature for day t, touch only data from day <= t-1.
    return f

features = build_features(px)
features.tail()


In [ ]:
# PROVIDED — look-ahead detector. Do not modify.
def assert_no_lookahead(make_features, prices, t0_frac=0.6, seed=0):
    """Scramble the FUTURE (rows after t0) and confirm features for rows up to t0 do not
    change. A past-only feature cannot depend on future data, so the max change must be ~0.
    Catches forward shifts, centered windows, and full-sample statistics."""
    base = make_features(prices)
    pert = prices.copy()
    t0 = int(len(prices) * t0_frac)
    rng = np.random.default_rng(seed)
    fut = pert.iloc[t0 + 1:].to_numpy()
    pert.iloc[t0 + 1:] = fut * rng.uniform(0.5, 1.5, size=fut.shape)   # scramble the future
    pf = make_features(pert)
    idx = base.index[:t0 + 1]
    diff = (base.loc[idx] - pf.loc[idx, base.columns]).abs().to_numpy()
    diff = diff[np.isfinite(diff)]
    return float(diff.max()) if diff.size else 0.0

disc = assert_no_lookahead(build_features, px)
print(f'Max change in past features when the future is scrambled: {disc:.2e}')
assert disc < 1e-8, 'LEAKAGE: a feature depends on the future. Look for a missing shift, a centered window, or a full-sample statistic.'
print('PASS - features use only past information.')


*C3 written answer:* TODO — explain in a paragraph why your features pass the check.


In [ ]:
# C2: target + C4: time-ordered split.
# The comparison must preserve the unavailable final label as NaN. In pandas,
# `(NaN > 0)` is False, so converting that comparison directly to int would
# silently invent a final "down" observation.
next_ret = px['log_ret'].shift(-1)
target = (next_ret > 0).where(next_ret.notna()).rename('y')

data = features.join(target).dropna()
cut = int(len(data) * 0.7)
train, test = data.iloc[:cut], data.iloc[cut:]
print(f'Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} rows)')
print(f'Test:  {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} rows)')
print('No shuffling: the test period is strictly after the train period.')


## Part D — Baseline & reflection (20 pts)


In [ ]:
# D1: naive baseline on the TEST period (majority class / 'predict up').
y_test = test['y']
baseline_acc = max(y_test.mean(), 1 - y_test.mean())
print(f'Naive baseline accuracy on test: {baseline_acc:.3%}')
# (No model required in HW1 — just the baseline. Models start in Week 4.)


*D2 reflection (250–400 words):* TODO
- Which Week 2 stylized facts did your data exhibit?
- Where could leakage have crept in, and how did you prevent it?
- What would make a model trained on this data fail in reality?


## Extensions (optional)

The required core is everything above, and the self-check tests only that. These extensions are optional and **not** required for full marks — a clean, well-reasoned core beats a sprawling one. If you want to go further:

- Add a second feature family (calendar/seasonality, or a range/RSI variant) and confirm it passes the look-ahead check.
- Repeat the Week-2 stylized-facts diagnostics on a second instrument and compare.
- Swap the 70/30 split for a purged/embargoed split and note what changes.

## Self-check (auto-graded)

Run this cell **last**, after the whole notebook. Every line must print **PASS** before you submit — it verifies the mechanical requirements (reproducibility, leak-free features, time-ordered splits, metrics computed). Your written answers and judgment are graded by hand.

In [ ]:
# ================= SELF-CHECK (auto-graded) =================
# Run LAST. Each line reports PASS/FAIL for a mechanical requirement.
# Written answers / judgment are graded separately, by hand.
def _check(name, fn, hint=""):
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ") + name + ("" if ok else "   -> " + hint))
    return ok
_r = []
_r.append(_check("reproducibility: SEED set", lambda: SEED is not None, "set SEED = 42 in the setup cell"))
_r.append(_check("returns computed (A2)", lambda: px['log_ret'].notna().sum() > 100 or px['simple_ret'].notna().sum() > 100, "compute px['log_ret'] or px['simple_ret'] in Part A2"))
_r.append(_check(">= 6 numeric leak-free features (C1)", lambda: features.select_dtypes('number').shape[1] >= 6, "add features in build_features() until `features` has >= 6 numeric columns"))
_r.append(_check("no look-ahead (C3)", lambda: assert_no_lookahead(build_features, px) < 1e-8, "a feature sees the future: add a .shift(), avoid centered windows and full-sample stats"))
_r.append(_check("unavailable final target remains missing (C2)", lambda: pd.isna(target.iloc[-1]), "do not let `(NaN > 0).astype(int)` invent a final label"))
_r.append(_check("time-ordered, strictly disjoint split (C4)", lambda: train.index.max() < test.index.min(), "the test period must start after train ends"))
_r.append(_check("naive baseline on TEST (D1)", lambda: 0.0 < float(baseline_acc) < 1.0, "compute baseline_acc on the test period"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r), "Fix the FAIL items above before submitting."


## AI-use disclosure (required)
TODO: State specifically whether and how you used AI tools. You remain responsible for every
claim, decision, line of code, citation, and interpretation, and you must be able to explain them.
If you did not use AI, say so.

## Reproducibility checklist
- [ ] Runs with Restart & Run All  - [ ] Seed set  - [ ] Train/test dates stated
- [ ] Data source documented  - [ ] Packages recoverable
